In [0]:
cancer_treatment_comment = "This table captures comprehensive treatment for breast cancer patients. It includes details on systemic therapies (type, drugs, intent, cycles, start/end dates, and reasons for stopping), as well as radiotherapy administration (sites, dose, fractions, and boost)."

schema_cancer_treatment = StructType([

    StructField(
        "PERSON_ID",
        LongType(),
        True,
        metadata={"comment": "This is the value of the unique primary identifier of the PERSON table. It is an internal system assigned number."}
    ),

    StructField(
        "NHS_Number",
        StringType(),
        True,
        metadata={"comment": "The NHS NUMBER, the primary identifier of a PERSON, is a unique identifier for a PATIENT within the NHS in England and Wales. Based on this field we identify the COHORT patients from the DWH"}
    ),

    StructField(
        "MRN",
        StringType(),
        True,
        metadata={"comment": "Primary identifier of PERSON in iQEMO system; joined with BHResearch Demographics table on MRN"}
    ),

    StructField(
        "start_date",
        DateType(),
        True,
        metadata={"comment": "Date on which the first administration of the course is to be done./The Start Date of the treatment course."}
    ),

    StructField(
        "drug",
        StringType(),
        True,
        metadata={"comment": "Standardised drug name"}
    ),

    # ---------------------------
    # ARIA fields
    # ---------------------------
    StructField(
        "DosageForm",
        StringType(),
        True,
        metadata={"comment": "Code used to identify the dosage form of the agent from ARIA."}
    ),

    StructField(
        "DoseLevel",
        StringType(),
        True,
        metadata={"comment": "Code to identify the dose level of the agent. The field is used to create many forms of a specific agent normally based on the dosage to be given. i.e. a pediatric form, high, medium, low, etc from ARIA"}
    ),

    StructField(
        "AdmnDosageUnit",
        StringType(),
        True,
        metadata={"comment": "Dosage unit admninstered"}
    ),

    StructField(
        "AdmnRoute",
        StringType(),
        True,
        metadata={"comment": "Code to identify the route which should be used to administer the agent."}
    ),

    StructField(
        "RxDose",
        IntegerType(),
        True,
        metadata={"comment": "The amount of the agent in this prescription to order from the pharmacy with the intent of administering it."}
    ),

    StructField(
        "RxTotal",
        IntegerType(),
        True,
        metadata={"comment": "The total amount of the agent which will be ordered for the patient according to the frequency and amount for this agent item."}
    ),

    # ---------------------------
    # IQEMO fields
    # ---------------------------
    StructField(
        "RegimenName",
        StringType(),
        True,
        metadata={"comment": "Name for this Regimen. Must be unique within the organisation."}
    ),

    StructField(
        "DefaultCycles",
        IntegerType(),
        True,
        metadata={"comment": "The default number of cycles to create when booking a course of this regimen."}
    ),

    StructField(
        "ChemoRadiation",
        BooleanType(),
        True,
        metadata={"comment": "indicates if given with radiotherapy."}
    ),

    StructField(
        "OPCSProcurementCode",
        StringType(),
        True,
        metadata={"comment": "The NHS OPCS procurement code for this item."}
    ),

    StructField(
        "OPCSDeliveryCode",
        StringType(),
        True,
        metadata={"comment": "The NHS OPCS delivery code for this item."}
    ),

    StructField(
        "Indication",
        StringType(),
        True,
        metadata={"comment": "A free text description of the Indication for this regimen. Used to detail appropriate usage and displayed when booking courses of this regimen."}
    ),

    StructField(
        "EndDate",
        DateType(),
        True,
        metadata={"comment": "The End Date of the course. Set by the stored procedure psp_ChemotherapyCourse_UpdateEndDate."}
    ),

    StructField(
        "FinalTreatmentDate",
        DateType(),
        True,
        metadata={"comment": "The final day of treatment. Not the same as EndDate which is when the actual treatment will end including TTOs etc"}
    ),

    StructField(
        "CourseFinished",
        BooleanType(),
        True,
        metadata={"comment": "Indicates if this course is finished. This field is updated as part of a nightly maintenance task that executes psp_ChemotherapyCourse_UpdateFinished"}
    ),

    StructField(
        "PlannedCycles",
        IntegerType(),
        True,
        metadata={"comment": "The number of cycles that were planned. Required for NHS SACT report."}
    ),

    StructField(
        "CycleCancelledFrom",
        IntegerType(),
        True,
        metadata={"comment": "If the remainder of the course was cancelled this indicates the cycle that the course was cancelled from."}
    ),

    StructField(
        "record_type",
        StringType(),
        True,
        metadata={"comment": "Record source: matched / aria_only / iqemo_only"}
    ),
    StructField(
        "ADC_UPDT", 
        TimestampType(), 
        True, 
        metadata= {"comment": "Timestamp of last update."}
    )

])

# DRUG CLEAN FUNCTION

def clean_drug_names(text):
    if not isinstance(text, str):
        return []
    
    # Normalise the text
    text = text.lower()
    text = re.sub(r"[^\w\s+/]", " ", text)
    # Remove salt from the drugs
    salts = [
        "hydrochloride", "sulphate", "sulfate",
        "phosphate", "acetate", "succinate",
        "sodium", "disodium"
    ]
    for s in salts:
        text = text.replace(s, "")
    # Replace brand names -- needs to be check with omop relationship to check the drug name and drug brand names
    synonyms = {
        "abraxane": "paclitaxel",
        "herceptin": "trastuzumab",
        "herzuma": "trastuzumab",
        "caelyx": "doxorubicin",
        "phesgo": "pertuzumab trastuzumab",
    }
    for k, v in synonyms.items():
        text = text.replace(k, v)

    drugs = re.split(r"[+/]", text)
    return [d.strip() for d in drugs if d.strip() and "placebo" not in d]


clean_udf = udf(clean_drug_names, ArrayType(StringType()))

def cancer_treatment_incr():

    max_adc_updt = get_max_timestamp("4_prod.bronze.map_cancer_treatment")

    # ARIA

    pt_inst_key = spark.table("4_prod.raw.aria_pt_inst_key").alias("Ptkey")
    agt_rx = spark.table("4_prod.raw.aria_agt_rx").alias("Arx")
    rx = spark.table("4_prod.raw.aria_rx").alias("Rx")

    person_alias = (
        spark.table("4_prod.raw.mill_person_alias")
        .withColumn(
            "PERSON_ALIAS_TYPE_DESC",
            when(col("PERSON_ALIAS_TYPE_CD") == 18, "NHS_number")
            .when(col("PERSON_ALIAS_TYPE_CD") == 10, "MRN")
        )
        .alias("PA")
    )

    aria = (
        pt_inst_key
        .join(agt_rx, col("Arx.pt_id") == col("Ptkey.pt_id"), "inner")
        .join(rx, (col("Arx.pt_id") == col("Rx.pt_id")) & (col("Arx.rx_id") == col("Rx.rx_id")), "inner")
        .join(
            person_alias,
            (regexp_replace(col("Ptkey.pt_key_value"), ' ', '') == col("PA.ALIAS")) &
            (col("PA.PERSON_ALIAS_TYPE_CD").isin(18, 10)),
            "left"
        )
        .filter(col("PERSON_ALIAS_TYPE_DESC").isNotNull()))


    aria = (
        aria
        .groupBy(
            col("PA.PERSON_ID"),
            col("Arx.admn_start_date"),
            col("Arx.tp_name"),
            col("Arx.AGT_NAME"),
            col("Arx.dosage_form"),
            col("Arx.dose_level"),
            col("Arx.admn_dosage_unit"),
            col("Arx.admn_route"),
            col("Arx.rx_dose"),
            col("Arx.rx_total")
        )
        .agg(
            first(
                when(col("PA.PERSON_ALIAS_TYPE_DESC") == "NHS_number", col("PA.ALIAS")),
                ignorenulls=True
            ).alias("NHS_Number"),

            first(
                when(col("PA.PERSON_ALIAS_TYPE_DESC") == "MRN", col("PA.ALIAS")),
                ignorenulls=True
            ).alias("MRN")
        )
    )

    aria_final = (aria
        .withColumn("aria_clean_drugs", clean_udf(col("AGT_NAME")))
        .withColumn("drug", explode(col("aria_clean_drugs")))
        .withColumn("start_date", to_date(col("admn_start_date")))
        .select(
            col("PA.PERSON_ID").alias("PERSON_ID"),
            col("NHS_Number").alias("NHS_Number"),
            col("MRN").alias("MRN"),
            col("start_date"),
            col("Arx.AGT_NAME").cast(StringType()).alias("ProductDesc"),
            col("Arx.dosage_form").alias("DosageForm"),
            col("Arx.dose_level").alias("DoseLevel"),
            col("Arx.admn_dosage_unit").alias("AdmnDosageUnit"),
            col("Arx.admn_route").alias("AdmnRoute"),
            col("Arx.rx_dose").cast(IntegerType()).alias("RxDose"),
            col("Arx.rx_total").cast(IntegerType()).alias("RxTotal"),
            col("drug")
        )
        .filter(col("PERSON_ID").isNotNull())
        .dropDuplicates()
    )

    # IQEMO

    treatment_cycle = spark.table("4_prod.raw.iqemo_treatment_cycle").alias("TC")
    chemotherapy_course = spark.table("4_prod.raw.iqemo_chemotherapy_course").alias("CC")
    regimen = spark.table("4_prod.raw.iqemo_regimen").alias("RG")
    iqemo_patient = spark.table("4_prod.raw.iqemo_patient").alias("PT")
    person_alias = spark.table("4_prod.raw.mill_person_alias").alias("PA")

    iqemo = (
        treatment_cycle
        .join(chemotherapy_course, col("CC.ChemoTherapyCourseID") == col("TC.ChemoTherapyCourseID"), "left")
        .join(regimen, col("CC.RegimenID") == col("RG.RegimenID"), "left")
        .join(iqemo_patient, col("TC.PatientID") == col("PT.PatientID"), "left")
        .join(
            person_alias,
            (trim(col("PT.PrimaryIdentifier")) == trim(col("PA.ALIAS"))) &
            (col("PA.PERSON_ALIAS_TYPE_CD") == 10) &
            (col("PA.ALIAS_POOL_CD").isin(683996, 1115132483, 6200990, 6173940)),
            "left"
        )
        # add drug parsing
        .withColumn("iqemo_clean_drugs", clean_udf(col("SactName")))
        .withColumn("drug", explode(col("iqemo_clean_drugs")))
        .select(
            col("PA.PERSON_ID").alias("PERSON_ID"),
            col("PT.PrimaryIdentifier").alias("MRN"),
            col("PT.NHSNumber").alias("NHS_Number"),
            # col("TC.TreatmentCycleID").alias("TreatmentCycleID"),
            # col("TC.PrescribedDate").alias("PrescribedDate"),
            # col("TC.TemplateName").alias("TemplateName"),
            col("RG.Name").alias("RegimenName"),
            col("RG.DefaultCycles").alias("DefaultCycles"),
            col("RG.ChemoRadiation").alias("ChemoRadiation"),
            col("RG.OPCSProcurementCode").alias("OPCSProcurementCode"),
            col("RG.OPCSDeliveryCode").alias("OPCSDeliveryCode"),
            # col("RG.SactName").alias("SactName"),
            col("RG.Indication").alias("Indication"),
            col("CC.StartDate").alias("start_date"),
            col("CC.EndDate").alias("EndDate"),
            col("CC.FinalTreatmentDate").alias("FinalTreatmentDate"),
            col("CC.CourseFinished").alias("CourseFinished"),
            col("CC.PlannedCycles").alias("PlannedCycles"),
            col("CC.CycleCancelledFrom").alias("CycleCancelledFrom"),
            # col("CC.RegimenName").alias("CourseRegimenName"),
            # col("CC.RegimenID").alias("CourseRegimenID"),
            # col("CC.RegimenNumber").alias("RegimenNumber")
            col("drug")
        )
        .filter(col("PERSON_ID").isNotNull())
        .dropDuplicates()
    )

    # ================================
    # MATCH + DEDUP
    # ================================
    full_data = (
        aria_final.alias("ar")
        .join(
            iqemo.alias("iq"),
            (
                    (col("ar.PERSON_ID") == col("iq.PERSON_ID")) |
                    (col("ar.MRN") == col("iq.MRN")) |
                    (col("ar.NHS_Number") == col("iq.NHS_Number"))
            ),
            "full_outer")
        )


    full_data = (
        full_data
        .withColumn(
            "date_diff",
            abs(datediff(col("ar.start_date"), col("iq.start_date")))
        )
        .withColumn(
            "drug_distance",
            levenshtein(col("ar.drug"), col("iq.drug"))
        )
        .withColumn(
            "drug_similarity",
            1 - col("drug_distance") / greatest(length(col("iq.drug")), lit(1))
        )
    )
    # Identify the matched records
    matched = (
        full_data
        .filter(
            (col("drug_similarity") >= 0.8) &
            (col("date_diff") <= 3)
        )
    )

    # Get Unmatched records

    # Aria

    unmatched_aria = (
        aria_final.alias("ar")
        .join(
            matched.select(
                col("ar.PERSON_ID").alias("PERSON_ID"),
                col("ar.start_date").alias("start_date"),
                col("ar.drug").alias("drug")
            ).distinct(),
            ["PERSON_ID", "start_date", "drug"],
            "left_anti"
        )
        .withColumn("record_type", lit("aria_only"))
    )

    # Iqemo
    unmatched_iqemo = (
        iqemo.alias("iq")
        .join(
            matched.select(
                col("iq.PERSON_ID").alias("PERSON_ID"),
                col("iq.start_date").alias("start_date"),
                col("iq.drug").alias("drug")
            ).distinct(),
            ["PERSON_ID", "start_date", "drug"],
            "left_anti"
        )
        .withColumn("record_type", lit("iqemo_only"))
    )


    matched_final = (
        matched
        .select(
            coalesce(col("ar.PERSON_ID"), col("iq.PERSON_ID")).alias("PERSON_ID"),
            coalesce(col("ar.NHS_Number"), col("iq.NHS_Number")).alias("NHS_Number"),
            coalesce(col("ar.MRN"), col("iq.MRN")).alias("MRN"),

            coalesce(col("ar.start_date"), col("iq.start_date")).alias("start_date"),
            coalesce(col("ar.drug"), col("iq.drug")).alias("drug"),

            # ARIA-specific fields
            col("ar.DosageForm"),
            col("ar.DoseLevel"),
            col("ar.AdmnDosageUnit"),
            col("ar.AdmnRoute"),
            col("ar.RxDose"),
            col("ar.RxTotal"),

            # IQEMO-specific fields
            col("iq.RegimenName"),
            col("iq.DefaultCycles"),
            col("iq.ChemoRadiation"),
            col("iq.OPCSProcurementCode"),
            col("iq.OPCSDeliveryCode"),
            col("iq.Indication"),
            col("iq.EndDate"),
            col("iq.FinalTreatmentDate"),
            col("iq.CourseFinished"),
            col("iq.PlannedCycles"),
            col("iq.CycleCancelledFrom"),
            lit("matched").alias("record_type")
        )
    )

    final_df = (
        matched_final
        .unionByName(unmatched_aria, allowMissingColumns=True)
        .unionByName(unmatched_iqemo, allowMissingColumns=True)
        .dropDuplicates()
        .withColumn("ADC_UPDT", current_timestamp())
        .filter(col("ADC_UPDT") > max_adc_updt)
    )

    return final_df

updates_df = cancer_treatment_incr()

# Validate casting across all columns  
update_table(updates_df, "8_dev.bronze.map_cancer_treatment", ["Person_ID","start_date","drug"], schema_cancer_treatment, cancer_treatment_comment)